# 05 — Classical Training and Tuning

Train baseline SVM, Random Forest, K-NN, and optionally XGBoost on classical features, then optimize with GridSearchCV (5-fold CV). Saves models, metrics, and confusion matrices for comparison.

In [1]:
# Imports and setup
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# XGBoost is optional — handle missing package gracefully
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as e:
    print('XGBoost not available, skipping XGBoost models:', e)
    XGB_AVAILABLE = False

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
FEAT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'classical_features'
MODEL_DIR = PROJECT_ROOT / 'models' / 'classical'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
print('Project root:', PROJECT_ROOT)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4


In [1]:
import mlflow
import mlflow.sklearn
from datetime import datetime

# Set the experiment name
mlflow.set_experiment("BikeAI_Classical_Models")

# Optional: Set tracking URI if you want mlruns in a specific location
# mlflow.set_tracking_uri("file:./mlruns")


/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2025/11/27 17:35:30 INFO mlflow.tracking.fluent: Experiment with name 'BikeAI_Classical_Models' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/harryirving/Development/projects/ai-ml/BikeAIv4/notebooks/mlruns/903988454064504742', creation_time=1764264930456, experiment_id='903988454064504742', last_update_time=1764264930456, lifecycle_stage='active', name='BikeAI_Classical_Models', tags={}>

In [2]:
# Load config
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
train_cfg = (cfg.get('training') or cfg.get('config', {}).get('training') or {})
TEST_SIZE = float(train_cfg.get('test_size', 0.2))
VAL_SIZE = float(train_cfg.get('val_size', 0.1))
RANDOM_STATE = int(train_cfg.get('random_state', 42))
print(f'Test size: {TEST_SIZE}, Val size: {VAL_SIZE}, Random state: {RANDOM_STATE}')


Test size: 0.2, Val size: 0.1, Random state: 42


In [3]:
# Load features
feat_csv = FEAT_DIR / 'classical_features.csv'
df = pd.read_csv(feat_csv)
print('Loaded features:', df.shape)

label_col = 'label'
path_col = 'path'
feature_cols = [c for c in df.columns if c not in [label_col, path_col]]

# Clean feature matrix (handle NaN/Inf robustly)
X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.values
y_labels = df[label_col].values

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_labels)
joblib.dump(le, MODEL_DIR / 'label_encoder.pkl')

print('Classes:', list(le.classes_))
print('Feature matrix:', X.shape)
print('Labels:', y.shape, 'unique:', np.unique(y))

Loaded features: (19044, 366)
Classes: ['angle_grinder', 'background', 'tools']
Feature matrix: (19044, 364)
Labels: (19044,) unique: [0 1 2]


In [4]:
# Train/Val/Test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
val_fraction = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_fraction, stratify=y_train_val, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

Train: (13330, 364) Val: (1905, 364) Test: (3809, 364)


In [5]:
# Standardization
scaler = StandardScaler()
scaler.fit(X_train)
X_train_sc = scaler.transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')
print('Scaler saved.')

Scaler saved.


In [6]:
# Baseline models
models_baseline = {
    'SVM': SVC(random_state=RANDOM_STATE, probability=True),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline_results = []

for name, model in models_baseline.items():
    print(f'\nTraining {name} (baseline)...')
    model.fit(X_train_sc, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val_sc))
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='accuracy')
    baseline_results.append({
        'model': name,
        'val_accuracy': val_acc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    })
    print(f'{name} — Val Acc: {val_acc:.4f}, CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    joblib.dump(model, MODEL_DIR / f'{name.lower()}_baseline.pkl')

df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv(METRICS_DIR / 'baseline_results.csv', index=False)
print('\nBaseline results saved.')


Training SVM (baseline)...
SVM — Val Acc: 0.9879, CV: 0.9865 ± 0.0013

Training RandomForest (baseline)...
RandomForest — Val Acc: 0.9932, CV: 0.9896 ± 0.0025

Training KNN (baseline)...
KNN — Val Acc: 0.9890, CV: 0.9881 ± 0.0010

Training XGBoost (baseline)...
XGBoost — Val Acc: 0.9974, CV: 0.9931 ± 0.0012

Baseline results saved.


In [7]:
param_grids = {
    'SVM': {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.001, 0.01],
        'kernel': ['rbf']
    },
    'RandomForest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5],
        'max_features': ['sqrt', 'log2']
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 11, 15, 21],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }
}


In [8]:
tuned_results = []
best_models = {}

for name in models_baseline.keys():
    print(f'\n{"="*60}')
    print(f'Tuning {name} with GridSearchCV...')
    print(f'{"="*60}')
    
    base_model = models_baseline[name]
    grid = GridSearchCV(
        base_model, param_grids[name], cv=cv, scoring='accuracy', 
        n_jobs=-1, verbose=1
    )
    grid.fit(X_train_sc, y_train)
    
    best_model = grid.best_estimator_
    val_acc = accuracy_score(y_val, best_model.predict(X_val_sc))
    
    tuned_results.append({
        'model': name,
        'best_params': json.dumps(grid.best_params_),
        'train_cv_score': grid.best_score_,
        'val_accuracy': val_acc
    })
    
    print(f'\nBest params: {grid.best_params_}')
    print(f'Train CV score: {grid.best_score_:.4f}')
    print(f'Val accuracy: {val_acc:.4f}')
    
    joblib.dump(best_model, MODEL_DIR / f'{name.lower()}_tuned.pkl')
    best_models[name] = best_model

df_tuned = pd.DataFrame(tuned_results)
df_tuned.to_csv(METRICS_DIR / 'tuned_results.csv', index=False)
print('\nTuned models saved.')



Tuning SVM with GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Train CV score: 0.9923
Val accuracy: 0.9948

Tuning RandomForest with GridSearchCV...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best params: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Train CV score: 0.9897
Val accuracy: 0.9932

Tuning KNN with GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best params: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}
Train CV score: 0.9944
Val accuracy: 0.9963

Tuning XGBoost with GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fits


/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [23:41:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [23:41:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/xgboost/training.py:199: UserWarning: [23:41:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/xgboost/training.


Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
Train CV score: 0.9950
Val accuracy: 0.9963

Tuned models saved.


In [9]:
test_results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred)
    test_results.append({'model': name, 'test_accuracy': acc})
    print(f'\n{name} Test Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{name} — Confusion Matrix')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'cm_{name.lower()}_tuned.png', dpi=150)
    plt.close()

df_test = pd.DataFrame(test_results)
df_test.to_csv(METRICS_DIR / 'test_results_classical.csv', index=False)
print('\nTest results saved.')



SVM Test Accuracy: 0.9945
               precision    recall  f1-score   support

angle_grinder       0.99      1.00      0.99      1276
   background       1.00      1.00      1.00      1614
        tools       1.00      0.98      0.99       919

     accuracy                           0.99      3809
    macro avg       0.99      0.99      0.99      3809
 weighted avg       0.99      0.99      0.99      3809


RandomForest Test Accuracy: 0.9892
               precision    recall  f1-score   support

angle_grinder       0.98      1.00      0.99      1276
   background       1.00      0.99      1.00      1614
        tools       0.99      0.97      0.98       919

     accuracy                           0.99      3809
    macro avg       0.99      0.99      0.99      3809
 weighted avg       0.99      0.99      0.99      3809


KNN Test Accuracy: 0.9942
               precision    recall  f1-score   support

angle_grinder       0.99      1.00      0.99      1276
   background       1.0

In [10]:
comparison = df_baseline.merge(df_tuned, on='model', suffixes=('_baseline', '_tuned'))
comparison = comparison.merge(df_test, on='model')
comparison.to_csv(METRICS_DIR / 'classical_model_comparison.csv', index=False)

print('\n' + '='*60)
print('CLASSICAL MODEL COMPARISON')
print('='*60)
print(comparison[['model', 'val_accuracy_baseline', 'val_accuracy_tuned', 'test_accuracy']])



CLASSICAL MODEL COMPARISON
          model  val_accuracy_baseline  val_accuracy_tuned  test_accuracy
0           SVM               0.987927            0.994751       0.994487
1  RandomForest               0.993176            0.993176       0.989236
2           KNN               0.988976            0.996325       0.994224
3       XGBoost               0.997375            0.996325       0.993962


In [11]:
# Check for file-level leakage
manifest = pd.read_csv(METRICS_DIR / 'preprocessing_manifest.csv')
if 'original_path' in manifest.columns and 'segment_path' in manifest.columns:
    # Map segment paths to original file stems
    seg_to_stem = dict(zip(manifest['segment_path'], manifest['original_path'].apply(lambda p: Path(str(p)).stem)))
    
    # Load test indices (assuming you saved split indices; if not, recreate split)
    # For now, assume current X_test corresponds to manifest rows (check alignment)
    test_paths = manifest['segment_path'].iloc[te_mask].values if 'te_mask' in locals() else []
    train_paths = manifest['segment_path'].iloc[tr_mask].values if 'tr_mask' in locals() else []
    
    # Get unique stems per set
    train_stems = set([seg_to_stem.get(str(p), '') for p in train_paths])
    test_stems = set([seg_to_stem.get(str(p), '') for p in test_paths])
    
    # Check overlap
    overlap = train_stems & test_stems
    print(f'Files in both train and test: {len(overlap)} / {len(test_stems)} test files')
    if len(overlap) > 0:
        print('LEAKAGE DETECTED: Segments from same files in train and test.')
        print('Example overlapping files:', list(overlap)[:5])
    else:
        print('No file-level leakage detected.')
else:
    print('Manifest incomplete—manual check needed.')


Files in both train and test: 0 / 0 test files
No file-level leakage detected.
